In [14]:
import pandas as pd
import numpy as np
import spiceypy as spy
import plotly.graph_objects as go
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.Utils import CanonicalUnits, GravitationalParameters

In [15]:
path_data = "../../datos/sbdb_query_results_NEOS.csv"

neos = pd.read_csv(path_data)
neos["q"]=neos["a"]*(1-neos["e"])
neos = neos[neos["q"] < 1.30]
neos

,pdes,epoch_mjd,a,e,i,om,w,ma,q
0,433,60200,1.458,0.2228,10.83,304.29,178.91,222.76,1.133158
1,719,60200,2.636,0.5470,11.58,183.85,156.23,56.29,1.194108
2,887,60200,2.472,0.5709,9.40,110.42,350.47,238.76,1.060735
3,1036,60200,2.666,0.5329,26.69,215.50,132.48,276.42,1.245289
4,1221,60200,1.919,0.4357,11.88,171.32,26.65,123.53,1.082892
...,...,...,...,...,...,...,...,...,...
34323,2024 CP5,60200,2.588,0.6057,3.48,309.59,206.24,321.35,1.020448
34324,2024 CQ5,60350,2.180,0.8506,2.16,228.85,146.50,16.09,0.325692
34325,2024 CR5,60352,1.136,0.3680,17.82,141.95,100.20,302.45,0.717952
34326,2024 CS5,60353,1.121,0.4184,42.92,140.26,168.63,222.21,0.651974


In [16]:
orbital_elements = np.column_stack((np.array(neos['q']), np.array(neos['e']), np.array(neos['i']), np.array(neos['om']), np.array(neos['w']), np.array(neos['ma'])))
orbital_elements

array([[1.1331576e+00, 2.2280000e-01, 1.0830000e+01, 3.0429000e+02,
        1.7891000e+02, 2.2276000e+02],
       [1.1941080e+00, 5.4700000e-01, 1.1580000e+01, 1.8385000e+02,
        1.5623000e+02, 5.6290000e+01],
       [1.0607352e+00, 5.7090000e-01, 9.4000000e+00, 1.1042000e+02,
        3.5047000e+02, 2.3876000e+02],
       ...,
       [7.1795200e-01, 3.6800000e-01, 1.7820000e+01, 1.4195000e+02,
        1.0020000e+02, 3.0245000e+02],
       [6.5197360e-01, 4.1840000e-01, 4.2920000e+01, 1.4026000e+02,
        1.6863000e+02, 2.2221000e+02],
       [9.5406220e-01, 6.6180000e-01, 4.6800000e+00, 1.8281000e+02,
        2.3497000e+02, 1.3025000e+02]])

In [17]:
deg = np.pi/180
AU_m = 1.496e11 #m
M_sun = 1.9891e30
G = 6.67430e-11 # m^3 / (kg s^2)
year = 365.25*24*3600 #s
mu = CanonicalUnits().mu
grav_params = GravitationalParameters(mu=mu)

In [18]:
state_vectors = np.zeros((len(orbital_elements), 6))
for index, elements in enumerate(orbital_elements):
    q = elements[0]
    e = elements[1]
    i = elements[2]*deg
    Omega = elements[3]*deg
    w = elements[4]*deg
    M = elements[5]*deg

    state_vector = spy.conics([q, e, i, Omega, w, M]+[0, mu], 0)
    state_vectors[index] = np.array(state_vector[:6])

In [19]:
state_vectors

array([[ 1.50564476e+00, -8.24086449e-01,  1.49155679e-01,
         1.48131740e+00,  4.01462703e+00,  6.66810679e-01],
       [-4.97321553e-01,  2.47466938e+00, -5.12774515e-01,
        -3.67123769e+00,  1.44378549e+00, -3.45684097e-01],
       [ 1.86254335e+00, -3.05239693e+00, -1.12659812e-01,
         1.39433630e+00,  2.01164521e+00, -3.32517926e-01],
       ...,
       [-8.27481539e-01,  6.47049584e-01,  1.55307411e-04,
        -1.66731661e+00, -5.86126182e+00,  1.81401780e+00],
       [-1.26385772e+00,  8.43129403e-01,  1.48471976e-01,
        -6.40888573e-01, -3.07389320e+00,  2.57901055e+00],
       [-3.19994846e+00, -3.05371847e+00,  2.36845340e-01,
         6.56921545e-01, -1.83756004e+00,  1.52884727e-01]])

In [20]:
neos_state_vectors = pd.DataFrame(state_vectors, columns=["x", "y", "z", "vx", "vy", "vz"])
neos_state_vectors

,x,y,z,vx,vy,vz
0,1.505645,-0.824086,0.149156,1.481317,4.014627,0.666811
1,-0.497322,2.474669,-0.512775,-3.671238,1.443785,-0.345684
2,1.862543,-3.052397,-0.112660,1.394336,2.011645,-0.332518
3,-2.775589,-1.576004,-0.165269,2.523834,-1.361705,1.294116
4,2.531814,-0.354390,-0.006681,1.364789,2.837460,-0.633405
...,...,...,...,...,...,...
34319,1.480841,1.473060,0.126486,-4.674638,0.771875,-0.189155
34320,-0.894754,0.749800,-0.044021,-7.018545,0.432904,-0.210074
34321,-0.827482,0.647050,0.000155,-1.667317,-5.861262,1.814018
34322,-1.263858,0.843129,0.148472,-0.640889,-3.073893,2.579011


In [21]:
joined = neos_state_vectors.join(neos)

In [22]:
joined 

,x,y,z,vx,vy,vz,pdes,epoch_mjd,a,e,i,om,w,ma,q
0,1.505645,-0.824086,0.149156,1.481317,4.014627,0.666811,433,60200.0,1.458,0.2228,10.83,304.29,178.91,222.76,1.133158
1,-0.497322,2.474669,-0.512775,-3.671238,1.443785,-0.345684,719,60200.0,2.636,0.5470,11.58,183.85,156.23,56.29,1.194108
2,1.862543,-3.052397,-0.112660,1.394336,2.011645,-0.332518,887,60200.0,2.472,0.5709,9.40,110.42,350.47,238.76,1.060735
3,-2.775589,-1.576004,-0.165269,2.523834,-1.361705,1.294116,1036,60200.0,2.666,0.5329,26.69,215.50,132.48,276.42,1.245289
4,2.531814,-0.354390,-0.006681,1.364789,2.837460,-0.633405,1221,60200.0,1.919,0.4357,11.88,171.32,26.65,123.53,1.082892
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34319,1.480841,1.473060,0.126486,-4.674638,0.771875,-0.189155,2024 CL5,60353.0,1.103,0.4445,4.24,154.06,101.15,300.81,0.612716
34320,-0.894754,0.749800,-0.044021,-7.018545,0.432904,-0.210074,2024 CM5,60200.0,1.867,0.5245,1.81,322.91,135.06,313.89,0.887759
34321,-0.827482,0.647050,0.000155,-1.667317,-5.861262,1.814018,2024 CN5,60353.0,1.623,0.3144,2.42,172.22,343.84,353.54,1.112729
34322,-1.263858,0.843129,0.148472,-0.640889,-3.073893,2.579011,2024 CO5,60352.0,1.708,0.3938,9.66,334.87,160.57,3.52,1.035390


In [ ]:
np.savetxt("NEOS_state_vectors.txt", state_vectors)

In [23]:
xs = state_vectors[:,0]
ys = state_vectors[:,1]
zs = state_vectors[:,2]
# Add 1 to shift the mean of the Gaussian distribution

fig = go.Figure()
fig.add_trace(go.Histogram(x=xs, name='x'))
fig.add_trace(go.Histogram(x=ys, name='y'))
fig.add_trace(go.Histogram(x=zs, name='z'))
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.update_xaxes(range=[-5, 5])
fig.show()

In [24]:
vxs = state_vectors[:,3]
vys = state_vectors[:,4]
vzs = state_vectors[:,5]
# Add 1 to shift the mean of the Gaussian distribution

fig = go.Figure()
fig.add_trace(go.Histogram(x=vxs, name='vx'))
fig.add_trace(go.Histogram(x=vys, name='vy'))
fig.add_trace(go.Histogram(x=vzs, name='vz'))
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.update_xaxes(range=[-10, 10])
fig.show()

In [ ]:
## km/s para entender el pico

In [ ]:
import numpy as np
import plotly.graph_objects as go
i = np.random.uniform(0,np.pi/2, 1000)
sini = np.sin(i)

fig = go.Figure()
fig.add_trace(go.Histogram(x=sini, name='sini'))


In [ ]:
u = np.random.uniform(-1,1,1000)
invsin = np.arcsin(2*u)

fig = go.Figure()
fig.add_trace(go.Histogram(x=invsin, name='arcsin'))

C:\Users\aguju\AppData\Local\Temp\ipykernel_21908\3016420775.py:2: RuntimeWarning:

invalid value encountered in arcsin



In [ ]:
I = np.linspace(np.pi/2, np.pi, 1000)
fI = 1 - (1/2)*np.sin(I)

fig = go.Figure()
fig.add_trace(go.Scatter(x=I, y=fI, mode='lines'))
fig.show()

#funcion cumulativa

In [ ]:
#queremos invertir la funcion cumulativa

u = np.random.uniform(0,1,100000)

mask = u <= 0.5
I1 = np.arcsin(2*u[mask])
I2 = np.pi - np.arcsin(2*(1-u[~mask]))

I = np.concatenate([I1, I2])

fig = go.Figure()
fig.add_trace(go.Histogram(x=I, name='I'))
fig.show()


In [31]:
u = np.random.uniform(0,1,100000)

mask = u <= 0.5
I1 = np.arcsin(2*u[mask])
I2 = np.pi - np.arcsin(2*(1-u[~mask]))

I = np.concatenate([I1, I2])

fig = go.Figure()
fig.add_trace(go.Histogram(x=I, name='I'))
fig.show()


In [ ]:
I = np.random

### Min and Max values from the real sample of NEAs

In [ ]:
x_min = np.min(state_vectors[:,0])
y_min = np.min(state_vectors[:,1])
z_min = np.min(state_vectors[:,2])
vx_min = np.min(state_vectors[:,3])
vy_min = np.min(state_vectors[:,4])
vz_min = np.min(state_vectors[:,5])

x_max = np.max(state_vectors[:,0])
y_max = np.max(state_vectors[:,1])
z_max = np.max(state_vectors[:,2])
vx_max = np.max(state_vectors[:,3])
vy_max = np.max(state_vectors[:,4])
vz_max = np.max(state_vectors[:,5])

print('minimun values: ', x_min, y_min, z_min, vx_min, vy_min, vz_min)
print('maximun values: ', x_max, y_max, z_max, vx_max, vy_max, vz_max)

minimun values:  -29.67099575993139 -9.482497383892479 -13.982881986808259 -12.255442606275022 -12.739226388575982 -8.225563609389805
maximun values:  13.743225654354687 6.571501861281292 19.0344038527165 16.97490727806321 16.7029347145099 7.943030013571505


In [26]:
joined

,x,y,z,vx,vy,vz,pdes,epoch_mjd,a,e,i,om,w,ma,q
0,1.505645,-0.824086,0.149156,1.481317,4.014627,0.666811,433,60200.0,1.458,0.2228,10.83,304.29,178.91,222.76,1.133158
1,-0.497322,2.474669,-0.512775,-3.671238,1.443785,-0.345684,719,60200.0,2.636,0.5470,11.58,183.85,156.23,56.29,1.194108
2,1.862543,-3.052397,-0.112660,1.394336,2.011645,-0.332518,887,60200.0,2.472,0.5709,9.40,110.42,350.47,238.76,1.060735
3,-2.775589,-1.576004,-0.165269,2.523834,-1.361705,1.294116,1036,60200.0,2.666,0.5329,26.69,215.50,132.48,276.42,1.245289
4,2.531814,-0.354390,-0.006681,1.364789,2.837460,-0.633405,1221,60200.0,1.919,0.4357,11.88,171.32,26.65,123.53,1.082892
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34319,1.480841,1.473060,0.126486,-4.674638,0.771875,-0.189155,2024 CL5,60353.0,1.103,0.4445,4.24,154.06,101.15,300.81,0.612716
34320,-0.894754,0.749800,-0.044021,-7.018545,0.432904,-0.210074,2024 CM5,60200.0,1.867,0.5245,1.81,322.91,135.06,313.89,0.887759
34321,-0.827482,0.647050,0.000155,-1.667317,-5.861262,1.814018,2024 CN5,60353.0,1.623,0.3144,2.42,172.22,343.84,353.54,1.112729
34322,-1.263858,0.843129,0.148472,-0.640889,-3.073893,2.579011,2024 CO5,60352.0,1.708,0.3938,9.66,334.87,160.57,3.52,1.035390


In [27]:
filter_vz = joined[(joined['vz'] > 0) & (joined['vz'] < 0.2)]
filter_vz

,x,y,z,vx,vy,vz,pdes,epoch_mjd,a,e,i,om,w,ma,q
22,-3.465224,-0.003484,-0.105571,-0.206614,-2.299382,0.129334,2061,60200.0,2.2650,0.5359,3.80,207.36,157.08,167.64,1.051187
29,3.641772,0.066209,-0.153960,-0.611204,1.776847,0.046378,2201,60200.0,2.1780,0.7114,2.52,74.87,98.42,211.77,0.628571
30,-1.909181,-2.480063,0.426761,2.687336,-0.736809,0.039216,2202,60200.0,2.2910,0.5127,8.74,169.92,217.95,242.17,1.116404
33,-0.380442,0.942866,-0.102972,-3.838900,-3.965693,0.143087,2340,60200.0,0.8438,0.4499,5.86,211.30,40.09,264.78,0.464174
36,-1.342530,2.473812,-0.335986,-3.046862,-0.451224,0.128838,3102,60200.0,2.1510,0.4496,8.44,172.05,154.79,116.76,1.183910
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34308,-0.875324,0.657415,0.034284,-4.587671,-5.927706,0.185522,2024 CE4,60200.0,2.3920,0.7017,0.51,293.43,284.11,307.27,0.713534
34311,1.051194,1.910247,-0.096112,-4.168219,-0.476906,0.169222,2024 CH4,60352.0,1.4340,0.7075,12.07,321.55,67.26,27.89,0.419445
34312,-0.815757,0.643265,0.008664,-5.478913,-4.978626,0.136459,2024 CJ4,60348.0,2.4850,0.5594,2.27,90.90,49.78,0.57,1.094891
34316,1.590965,-0.283439,0.023175,-1.860303,4.924160,0.088674,2024 CQ4,60353.0,1.8630,0.4664,1.10,116.00,355.56,10.01,0.994097


In [29]:
import plotly.express as px
df = px.data.tips()
fig = px.histogram(joined, x="i")
fig.show()